# Phase 9 — Final End-to-End Adaptive Reasoning Benchmark

This phase performs the final unseen evaluation of the **frozen Phase 8 learned controller** inside the implemented latent Planner → Critic → Solver system.

### Frozen systems

- **Fixed-10:** Planner 10 / Critic 10 / Solver 10 latent steps.
- **Fixed-20:** Planner 20 / Critic 20 / Solver 20 latent steps.
- **Adaptive:** every agent reaches step 10; the learned Gradient Boosting Huber controller independently chooses **STOP at 10** or **CONTINUE to 20**.

### Final evaluation protocol

- Frozen Qwen/Qwen3-4B latent backbone and Phase 8 realignment artifact.
- Frozen Phase 8 controller and threshold `0.0`.
- 256 deterministically selected official GSM8K test problems.
- 256 deterministically selected official ARC-Challenge test problems.
- All unselected official test records remain labelled **SEALED_REMAINDER**.
- No controller training, feature selection, threshold tuning, or test-driven redesign occurs in this phase.
- True latent continuation is validated before the official test is opened: a 10-step rollout is continued by only the additional 10 latent steps, without replaying the role prompt or the first ten latent steps.

### Reporting added for Phase 9

The live console and saved outputs report correctness, Planner/Critic/Solver steps, generated tokens, prompt tokens, latency, GPU memory, adaptive STOP/CONTINUE decisions, compute tier, running accuracy and ETA. Final analysis includes Wilson accuracy intervals, exact McNemar tests, problem-level bootstrap confidence intervals, summary/prediction files, figures, SHA-256 manifests, resumable per-problem checkpoints, and compact recovery archives.

### Persistence

Before running this notebook, attach the locally saved **`adaptive-latent-mas_phase8_complete.zip`** as a Kaggle input. The notebook restores it automatically and creates a small **`adaptive-latent-mas_phase9_progress.zip`** during the run. If a session fails, attach the latest progress ZIP together with the Phase 8 master ZIP and rerun this same cell.

While the benchmark is running, progress can be checked from a Kaggle terminal with:

```bash
python /kaggle/working/adaptive-latent-mas/scripts/phase9_status.py
```


In [7]:
from pathlib import Path
import json
import shutil

INPUT = Path("/kaggle/input")
DEST = Path("/kaggle/working/adaptive-latent-mas")

candidates = []

for manifest in INPUT.rglob("phase8_final_manifest.json"):
    try:
        info = json.loads(manifest.read_text(encoding="utf-8"))
        root = manifest.parents[2]

        model = root / info["model"]["path"]
        realigner = root / "data/metadata/qwen3_4b_latent_realign.pt"

        if (
            info.get("status") == "PHASE_8_COMPLETE"
            and model.exists()
            and realigner.exists()
        ):
            candidates.append(root)

    except Exception:
        pass

if not candidates:
    raise FileNotFoundError(
        "No verified PHASE_8_COMPLETE project found in /kaggle/input."
    )

SOURCE = candidates[0]

if DEST.exists():
    shutil.rmtree(DEST)

shutil.copytree(SOURCE, DEST)

print("=" * 80)
print("PHASE 8 PROJECT RESTORED")
print("=" * 80)
print("Source :", SOURCE)
print("Master :", DEST)
print("Status : PHASE_8_COMPLETE")
print("Model  : OK")
print("Realigner: OK")
print("=" * 80)
print("NOW RUN THE PHASE 9 MAIN CELL")

PHASE 8 PROJECT RESTORED
Source : /kaggle/input/datasets/malkee/adaptive-latent-mas-phase8-complete
Master : /kaggle/working/adaptive-latent-mas
Status : PHASE_8_COMPLETE
Model  : OK
Realigner: OK
NOW RUN THE PHASE 9 MAIN CELL


In [8]:
# ==================================================================================================
# PHASE 9 — FINAL END-TO-END ADAPTIVE REASONING BENCHMARK
# One complete phase. No Phase 9A/9B/... subdivision.
# ==================================================================================================
from __future__ import annotations

import copy
import gc
import hashlib
import json
import math
import os
import shutil
import subprocess
import sys
import time
from dataclasses import asdict
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import yaml
from scipy.stats import binomtest
from transformers import AutoModelForCausalLM, AutoTokenizer

# --------------------------------------------------------------------------------------------------
# 0. FROZEN PHASE-9 SETTINGS
# --------------------------------------------------------------------------------------------------
SEED = 42
MODEL_ID = "Qwen/Qwen3-4B"
MODEL_REVISION = "1cfa9a7208912126459214e8b04321603b3df60c"
LOW_STEPS = 10
HIGH_STEPS = 20
FINAL_N_PER_DATASET = 256
MAX_NEW_TOKENS = 160
BOOTSTRAP_REPLICATES = 2000
CONTROLLER_THRESHOLD = 0.0
CONTINUATION_TOLERANCE = 1e-4
CHECKPOINT_EVERY = 20
PHASE9_VERSION = "1.0"

WORKING_ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
PROJECT_ROOT = WORKING_ROOT / "adaptive-latent-mas"
MASTER_ARCHIVE_NAME = "adaptive-latent-mas_phase8_complete.zip"
PROGRESS_ARCHIVE_NAME = "adaptive-latent-mas_phase9_progress.zip"
PROGRESS_ARCHIVE = WORKING_ROOT / PROGRESS_ARCHIVE_NAME

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# --------------------------------------------------------------------------------------------------
# 1. RESTORE THE SINGLE VERIFIED PHASE-8 MASTER PROJECT
# --------------------------------------------------------------------------------------------------
def phase8_complete(root: Path) -> bool:
    manifest = root / "experiments/summaries/phase8_final_manifest.json"
    if not manifest.exists():
        return False
    try:
        payload = json.loads(manifest.read_text(encoding="utf-8"))
        return payload.get("status") == "PHASE_8_COMPLETE"
    except Exception:
        return False

if not phase8_complete(PROJECT_ROOT):
    archives = list(INPUT_ROOT.rglob(MASTER_ARCHIVE_NAME))
    if not archives:
        raise FileNotFoundError(
            f"Attach {MASTER_ARCHIVE_NAME} as a Kaggle input before running Phase 9."
        )
    # Prefer the largest matching archive; ties are deterministic by path.
    archives = sorted(archives, key=lambda p: (p.stat().st_size, str(p)))
    master_archive = archives[-1]
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(master_archive), extract_dir=str(PROJECT_ROOT))
else:
    master_archive = None

if not phase8_complete(PROJECT_ROOT):
    raise RuntimeError("The restored project is not a verified PHASE_8_COMPLETE project.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PHASE8_MANIFEST_PATH = PROJECT_ROOT / "experiments/summaries/phase8_final_manifest.json"
PHASE8_MANIFEST = json.loads(PHASE8_MANIFEST_PATH.read_text(encoding="utf-8"))
CONTROLLER_PATH = PROJECT_ROOT / PHASE8_MANIFEST["model"]["path"]
ALIGNMENT_PATH = PROJECT_ROOT / "data/metadata/qwen3_4b_latent_realign.pt"

for required in [CONTROLLER_PATH, ALIGNMENT_PATH]:
    if not required.exists():
        raise FileNotFoundError(required)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(1024 * 1024)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

if sha256_file(CONTROLLER_PATH) != PHASE8_MANIFEST["model"]["sha256"]:
    raise RuntimeError("Phase 8 controller SHA-256 does not match its frozen manifest.")

# --------------------------------------------------------------------------------------------------
# 2. CREATE PHASE-9 DIRECTORIES + OPTIONAL PROGRESS RESTORE
# --------------------------------------------------------------------------------------------------
RUN_DIR = PROJECT_ROOT / "experiments/runs/phase9_final_evaluation"
EXAMPLE_DIR = RUN_DIR / "examples"
RESULT_FINAL_DIR = PROJECT_ROOT / "results/final"
TABLE_DIR = PROJECT_ROOT / "results/tables"
FIGURE_DIR = PROJECT_ROOT / "results/figures"
SUMMARY_DIR = PROJECT_ROOT / "experiments/summaries"
SCRIPT_DIR = PROJECT_ROOT / "scripts"
TEST_DIR = PROJECT_ROOT / "tests"

for directory in [RUN_DIR, EXAMPLE_DIR, RESULT_FINAL_DIR, TABLE_DIR, FIGURE_DIR, SUMMARY_DIR, SCRIPT_DIR, TEST_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# A previously downloaded progress archive can be attached to resume a failed Phase-9 run.
progress_inputs = sorted(INPUT_ROOT.rglob(PROGRESS_ARCHIVE_NAME), key=lambda p: (p.stat().st_size, str(p)))
if progress_inputs and not any(EXAMPLE_DIR.glob("*.json")):
    shutil.unpack_archive(str(progress_inputs[-1]), extract_dir=str(RUN_DIR))
    print("Recovered Phase 9 progress from:", progress_inputs[-1])

# --------------------------------------------------------------------------------------------------
# 3. SHORT TERMINAL STATUS COMMAND
# --------------------------------------------------------------------------------------------------
STATUS_SCRIPT = SCRIPT_DIR / "phase9_status.py"
STATUS_SCRIPT.write_text(r'''from pathlib import Path
import json

root = Path("/kaggle/working/adaptive-latent-mas")
run = root / "experiments/runs/phase9_final_evaluation"
status = run / "progress.json"
examples = run / "examples"

print("=" * 78)
print("PHASE 9 STATUS")
print("=" * 78)
print("Completed result files :", len(list(examples.glob("*.json"))) if examples.exists() else 0)
if status.exists():
    row = json.loads(status.read_text(encoding="utf-8"))
    for key in ["status", "completed", "total", "current_dataset", "current_example_id", "elapsed_hours", "eta_hours"]:
        if key in row:
            print(f"{key:<23}: {row[key]}")
    if "running_accuracy" in row:
        print("Running accuracy       :", row["running_accuracy"])
else:
    print("progress.json          : NOT YET CREATED")
print("=" * 78)
''', encoding="utf-8")

# --------------------------------------------------------------------------------------------------
# 4. IMPORT THE VERIFIED PHASE-8 IMPLEMENTATION
# --------------------------------------------------------------------------------------------------
from src.communication.latent_prompts import render_planner_prompt, render_critic_prompt, render_solver_prompt
from src.controllers.learned_marginal_utility import LearnedMarginalUtilityController
from src.datasets.loader import load_records
from src.evaluation.parsers import extract_prediction, is_correct
from src.experiments.fixed_latent_three_agent import ThreeAgentFixedLatentMASRunner
from src.latent.alignment import LatentRealigner
from src.latent.cache import past_length
from src.latent.latent_state_decoder import LatentStateDecoder
from src.latent.rollout import LatentRollout, LatentRolloutEngine, LatentStepTrace
from src.controllers.signals import predictive_entropy, representation_distance

# --------------------------------------------------------------------------------------------------
# 5. LOAD THE EXACT FROZEN QWEN BACKBONE REVISION
# --------------------------------------------------------------------------------------------------
if not torch.cuda.is_available():
    raise RuntimeError("Phase 9 requires a Kaggle GPU runtime.")

LATENT_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

class PinnedQwenBackend:
    def __init__(self, model_id: str, revision: str, dtype):
        self.model_id = model_id
        self.revision = revision
        self.device = "cuda"
        self.dtype = dtype
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            revision=revision,
            use_fast=True,
        )
        kwargs = dict(
            pretrained_model_name_or_path=model_id,
            revision=revision,
            device_map={"": 0},
        )
        try:
            self.model = AutoModelForCausalLM.from_pretrained(dtype=dtype, **kwargs)
        except TypeError:
            self.model = AutoModelForCausalLM.from_pretrained(torch_dtype=dtype, **kwargs)
        self.model.eval()
        for parameter in self.model.parameters():
            parameter.requires_grad_(False)
        self.output_embeddings = self.model.get_output_embeddings()

    @property
    def hidden_size(self):
        return int(self.model.config.hidden_size)

    def project_hidden_to_logits(self, hidden_state):
        return self.output_embeddings(hidden_state)

backend = PinnedQwenBackend(MODEL_ID, MODEL_REVISION, LATENT_DTYPE)
resolved_revision = getattr(backend.model.config, "_commit_hash", None)
if resolved_revision is not None and str(resolved_revision) != MODEL_REVISION:
    raise RuntimeError(f"Model revision mismatch: {resolved_revision} != {MODEL_REVISION}")

realigner = LatentRealigner(
    backend.model,
    cache_path=ALIGNMENT_PATH,
    regularization=1e-5,
    enabled=True,
)
realigner.build()
rollout_engine = LatentRolloutEngine(backend, realigner)
latent_decoder = LatentStateDecoder(backend)
adaptive_controller = LearnedMarginalUtilityController.load(
    CONTROLLER_PATH,
    threshold=CONTROLLER_THRESHOLD,
)

fixed10_runner = ThreeAgentFixedLatentMASRunner(
    backend,
    rollout_engine,
    latent_decoder,
    agent_budgets={"planner": 10, "critic": 10, "solver": 10},
    max_new_tokens=MAX_NEW_TOKENS,
)
fixed20_runner = ThreeAgentFixedLatentMASRunner(
    backend,
    rollout_engine,
    latent_decoder,
    agent_budgets={"planner": 20, "critic": 20, "solver": 20},
    max_new_tokens=MAX_NEW_TOKENS,
)

# --------------------------------------------------------------------------------------------------
# 6. TRUE LATENT CONTINUATION: STEP 10 -> STEP 20 WITHOUT REPLAY
# --------------------------------------------------------------------------------------------------
@torch.inference_mode()
def continue_latent_rollout(base_rollout: LatentRollout, extra_steps: int) -> LatentRollout:
    if extra_steps < 1:
        raise ValueError("extra_steps must be >= 1")

    past = base_rollout.past_key_values
    previous_hidden = base_rollout.last_hidden_state
    traces = list(base_rollout.traces)
    start_step = int(base_rollout.latent_steps)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.perf_counter()

    for offset in range(1, int(extra_steps) + 1):
        latent_vector = realigner(previous_hidden)
        latent_embedding = latent_vector.unsqueeze(1)
        cache_len = past_length(past)
        attention_mask = torch.ones(
            (latent_embedding.shape[0], cache_len + 1),
            dtype=torch.long,
            device=latent_embedding.device,
        )
        outputs = backend.model(
            inputs_embeds=latent_embedding,
            attention_mask=attention_mask,
            past_key_values=past,
            use_cache=True,
            output_hidden_states=True,
            return_dict=True,
        )
        past = outputs.past_key_values
        current_hidden = outputs.hidden_states[-1][:, -1, :]
        distance = representation_distance(current_hidden, previous_hidden)
        logits = backend.project_hidden_to_logits(current_hidden)
        entropy = predictive_entropy(logits)
        traces.append(
            LatentStepTrace(
                step=start_step + offset,
                representation_distance=float(distance.item()),
                entropy=float(entropy.item()),
                hidden_norm=float(current_hidden.float().norm(dim=-1).item()),
            )
        )
        previous_hidden = current_hidden

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    extra_latency = time.perf_counter() - start

    return LatentRollout(
        past_key_values=past,
        last_hidden_state=previous_hidden,
        prompt_tokens=int(base_rollout.prompt_tokens),
        latent_steps=start_step + int(extra_steps),
        latency_seconds=float(base_rollout.latency_seconds + extra_latency),
        initial_cache_length=int(base_rollout.initial_cache_length),
        final_cache_length=int(past_length(past)),
        traces=traces,
    )


def signals_first10(rollout: LatentRollout):
    if len(rollout.traces) < 10:
        raise RuntimeError("Controller requires ten latent traces.")
    return [
        {
            "step": int(trace.step),
            "representation_distance": float(trace.representation_distance),
            "entropy": float(trace.entropy),
            "hidden_norm": float(trace.hidden_norm),
        }
        for trace in rollout.traces[:10]
    ]


def controller_decision(role: str, rollout10: LatentRollout):
    started = time.perf_counter()
    result = adaptive_controller.decide(role, signals_first10(rollout10))
    return result, float(time.perf_counter() - started)

# --------------------------------------------------------------------------------------------------
# 7. VALIDATE TRUE CONTINUATION BEFORE OPENING THE OFFICIAL TEST SET
# --------------------------------------------------------------------------------------------------
training_probe = load_records("gsm8k", "train")[0]
probe_prompt = render_planner_prompt(backend, training_probe)
probe10 = rollout_engine.rollout(probe_prompt, latent_steps=10, past_key_values=None)
probe_continued = continue_latent_rollout(probe10, extra_steps=10)
probe20 = rollout_engine.rollout(probe_prompt, latent_steps=20, past_key_values=None)

trace_diffs = []
for a, b in zip(probe_continued.traces, probe20.traces):
    trace_diffs.extend([
        abs(a.representation_distance - b.representation_distance),
        abs(a.entropy - b.entropy),
        abs(a.hidden_norm - b.hidden_norm),
    ])
hidden_diff = float((probe_continued.last_hidden_state.float() - probe20.last_hidden_state.float()).abs().max().item())
trace_diff = float(max(trace_diffs) if trace_diffs else 0.0)
cache_match = int(probe_continued.final_cache_length) == int(probe20.final_cache_length)

CONTINUATION_VALIDATION = {
    "dataset": training_probe["dataset"],
    "example_id": training_probe["id"],
    "max_trace_absolute_difference": trace_diff,
    "max_final_hidden_absolute_difference": hidden_diff,
    "cache_length_match": bool(cache_match),
    "tolerance": CONTINUATION_TOLERANCE,
    "passed": bool(trace_diff <= CONTINUATION_TOLERANCE and hidden_diff <= CONTINUATION_TOLERANCE and cache_match),
}
(SUMMARY_DIR / "phase9_continuation_validation.json").write_text(
    json.dumps(CONTINUATION_VALIDATION, indent=2), encoding="utf-8"
)
if not CONTINUATION_VALIDATION["passed"]:
    raise RuntimeError(f"True 10->20 continuation validation failed: {CONTINUATION_VALIDATION}")

del probe10, probe_continued, probe20
gc.collect()
torch.cuda.empty_cache()

# --------------------------------------------------------------------------------------------------
# 8. ADAPTIVE THREE-AGENT RUNNER WITH TRUE CONTINUATION
# --------------------------------------------------------------------------------------------------
def compute_tier(total_steps: int) -> str:
    if total_steps <= 30:
        return "LOW"
    if total_steps <= 50:
        return "MEDIUM"
    return "HIGH"


@torch.inference_mode()
def run_adaptive(record):
    wall_start = time.perf_counter()
    controller_latency = 0.0

    planner_prompt = render_planner_prompt(backend, record)
    critic_prompt = render_critic_prompt(backend, record)
    solver_prompt = render_solver_prompt(backend, record)

    planner = rollout_engine.rollout(planner_prompt, latent_steps=LOW_STEPS, past_key_values=None)
    p_decision, p_ctrl_time = controller_decision("planner", planner)
    controller_latency += p_ctrl_time
    if p_decision["continue_reasoning"]:
        planner = continue_latent_rollout(planner, HIGH_STEPS - LOW_STEPS)

    critic = rollout_engine.rollout(
        critic_prompt,
        latent_steps=LOW_STEPS,
        past_key_values=planner.past_key_values,
    )
    c_decision, c_ctrl_time = controller_decision("critic", critic)
    controller_latency += c_ctrl_time
    if c_decision["continue_reasoning"]:
        critic = continue_latent_rollout(critic, HIGH_STEPS - LOW_STEPS)

    solver = rollout_engine.rollout(
        solver_prompt,
        latent_steps=LOW_STEPS,
        past_key_values=critic.past_key_values,
    )
    s_decision, s_ctrl_time = controller_decision("solver", solver)
    controller_latency += s_ctrl_time
    if s_decision["continue_reasoning"]:
        solver = continue_latent_rollout(solver, HIGH_STEPS - LOW_STEPS)

    decoded = latent_decoder.generate(
        last_hidden_state=solver.last_hidden_state,
        past_key_values=solver.past_key_values,
        max_new_tokens=MAX_NEW_TOKENS,
    )

    prediction = extract_prediction(record["dataset"], decoded["text"])
    correct = is_correct(record["dataset"], prediction, record["reference_answer"])
    p = int(planner.latent_steps)
    c = int(critic.latent_steps)
    s = int(solver.latent_steps)
    total_steps = p + c + s

    result = {
        "example_id": record["id"],
        "dataset": record["dataset"],
        "split": record["split"],
        "method": "three_agent_learned_adaptive_latent",
        "prediction": prediction,
        "reference": record["reference_answer"],
        "correct": bool(correct),
        "raw_output": decoded["text"],
        "planner_steps": p,
        "critic_steps": c,
        "solver_steps": s,
        "total_latent_steps": total_steps,
        "effective_latent_steps": total_steps,
        "actual_executed_latent_steps": total_steps,
        "latent_step_saving_vs_fixed20_percent": float(100.0 * (60 - total_steps) / 60.0),
        "compute_tier": compute_tier(total_steps),
        "generated_tokens": int(decoded["generated_tokens"]),
        "intermediate_generated_tokens": 0,
        "decode_prompt_tokens": int(decoded["decode_prompt_tokens"]),
        "prompt_tokens_total": int(planner.prompt_tokens + critic.prompt_tokens + solver.prompt_tokens),
        "latent_latency_seconds": float(planner.latency_seconds + critic.latency_seconds + solver.latency_seconds),
        "decode_latency_seconds": float(decoded["latency_seconds"]),
        "controller_latency_seconds": float(controller_latency),
        "model_component_latency_seconds": float(
            planner.latency_seconds + critic.latency_seconds + solver.latency_seconds + decoded["latency_seconds"]
        ),
        "wall_latency_seconds": float(time.perf_counter() - wall_start),
        "decoder_transition": decoded["transition"],
        "planner_decision": {
            "decision": p_decision["decision"],
            "predicted_marginal_utility": float(p_decision["predicted_marginal_utility"]),
        },
        "critic_decision": {
            "decision": c_decision["decision"],
            "predicted_marginal_utility": float(c_decision["predicted_marginal_utility"]),
        },
        "solver_decision": {
            "decision": s_decision["decision"],
            "predicted_marginal_utility": float(s_decision["predicted_marginal_utility"]),
        },
        "true_continuation": True,
    }
    return result


# Enrich the verified fixed-budget runner with Phase-9 reporting fields.
def run_fixed(runner, record, budget):
    wall_start = time.perf_counter()
    result = runner.run(record)
    result["effective_latent_steps"] = int(result["total_latent_steps"])
    result["actual_executed_latent_steps"] = int(result["total_latent_steps"])
    result["prompt_tokens_total"] = int(
        result["planner"]["prompt_tokens"] + result["critic"]["prompt_tokens"] + result["solver"]["prompt_tokens"]
    )
    result["latent_latency_seconds"] = float(
        result["planner"]["latent_latency_seconds"]
        + result["critic"]["latent_latency_seconds"]
        + result["solver"]["latent_latency_seconds"]
    )
    result["decode_latency_seconds"] = float(result["solver"]["decode_latency_seconds"])
    result["model_component_latency_seconds"] = float(result["latency_seconds"])
    result["wall_latency_seconds"] = float(time.perf_counter() - wall_start)
    result["controller_latency_seconds"] = 0.0
    result["compute_tier"] = "FIXED-10" if budget == 10 else "FIXED-20"
    result["latent_step_saving_vs_fixed20_percent"] = float(100.0 * (60 - result["total_latent_steps"]) / 60.0)
    return result

# --------------------------------------------------------------------------------------------------
# 9. LOAD OFFICIAL TEST DATA ONLY AFTER ALL METHODS ARE FROZEN
# --------------------------------------------------------------------------------------------------
gsm_test = load_records("gsm8k", "test")
arc_test = load_records("arc_challenge", "test")

for dataset_name, records in [("gsm8k", gsm_test), ("arc_challenge", arc_test)]:
    if len(records) < FINAL_N_PER_DATASET:
        raise RuntimeError(f"{dataset_name} official test set is too small.")
    ids = []
    for record in records:
        if record["dataset"] != dataset_name or str(record["split"]).lower() != "test":
            raise RuntimeError(f"Dataset-boundary violation in {dataset_name}.")
        ids.append(str(record["id"]))
    if len(ids) != len(set(ids)):
        raise RuntimeError(f"Duplicate test IDs in {dataset_name}.")


def deterministic_partition(records, dataset_name):
    ranked = sorted(
        records,
        key=lambda record: hashlib.sha256(
            f"PHASE9::{SEED}::{dataset_name}::{record['id']}".encode("utf-8")
        ).hexdigest(),
    )
    selected = ranked[:FINAL_N_PER_DATASET]
    sealed = ranked[FINAL_N_PER_DATASET:]
    return selected, sealed

selected_gsm, sealed_gsm = deterministic_partition(gsm_test, "gsm8k")
selected_arc, sealed_arc = deterministic_partition(arc_test, "arc_challenge")
selected_records = selected_gsm + selected_arc

selected_keys = {(r["dataset"], str(r["id"])) for r in selected_records}
sealed_keys = {(r["dataset"], str(r["id"])) for r in sealed_gsm + sealed_arc}
if selected_keys & sealed_keys:
    raise RuntimeError("Selected and SEALED_REMAINDER test boundaries overlap.")
if len(selected_keys) != FINAL_N_PER_DATASET * 2:
    raise RuntimeError("Incorrect Phase-9 selected-test size.")

boundary_manifest = {
    "phase": 9,
    "seed": SEED,
    "selection_rule": "SHA256 deterministic ranking independent of labels and outcomes",
    "official_test_counts": {"gsm8k": len(gsm_test), "arc_challenge": len(arc_test)},
    "final_evaluation_counts": {"gsm8k": len(selected_gsm), "arc_challenge": len(selected_arc)},
    "sealed_remainder_counts": {"gsm8k": len(sealed_gsm), "arc_challenge": len(sealed_arc)},
    "final_evaluation_ids": {
        "gsm8k": [str(r["id"]) for r in selected_gsm],
        "arc_challenge": [str(r["id"]) for r in selected_arc],
    },
    "sealed_remainder_id_sha256": {
        "gsm8k": hashlib.sha256("\n".join(str(r["id"]) for r in sealed_gsm).encode()).hexdigest(),
        "arc_challenge": hashlib.sha256("\n".join(str(r["id"]) for r in sealed_arc).encode()).hexdigest(),
    },
    "labels": {"selected": "FINAL_EVALUATION", "unselected": "SEALED_REMAINDER"},
}
BOUNDARY_PATH = RUN_DIR / "dataset_boundaries.json"
BOUNDARY_PATH.write_text(json.dumps(boundary_manifest, indent=2), encoding="utf-8")

# Deterministic mixed execution order across the two datasets.
selected_records = sorted(
    selected_records,
    key=lambda record: hashlib.sha256(
        f"PHASE9_ORDER::{SEED}::{record['dataset']}::{record['id']}".encode("utf-8")
    ).hexdigest(),
)

# --------------------------------------------------------------------------------------------------
# 10. RESUMABLE RESULT STORAGE
# --------------------------------------------------------------------------------------------------
def result_path(record):
    digest = hashlib.sha256(f"{record['dataset']}::{record['id']}".encode()).hexdigest()[:20]
    return EXAMPLE_DIR / f"{record['dataset']}__{digest}.json"


def _json_default(value):
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"Object of type {type(value).__name__} is not JSON serializable")


def atomic_json(path: Path, payload):
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False, default=_json_default),
        encoding="utf-8",
    )
    os.replace(tmp, path)


def validate_saved_result(payload):
    if payload.get("phase9_version") != PHASE9_VERSION:
        raise RuntimeError("Incompatible saved Phase-9 result version.")
    if payload.get("source_split") != "test":
        raise RuntimeError("Saved Phase-9 record is not from official test.")
    if set(payload.get("systems", {})) != {"fixed_10", "fixed_20", "adaptive"}:
        raise RuntimeError("Saved Phase-9 record does not contain all three systems.")
    if payload["systems"]["fixed_10"]["total_latent_steps"] != 30:
        raise RuntimeError("Saved fixed_10 result has invalid latent steps.")
    if payload["systems"]["fixed_20"]["total_latent_steps"] != 60:
        raise RuntimeError("Saved fixed_20 result has invalid latent steps.")
    adaptive_steps = int(payload["systems"]["adaptive"]["total_latent_steps"])
    if adaptive_steps not in {30, 40, 50, 60}:
        raise RuntimeError("Saved adaptive result has invalid latent-step total.")
    if (payload["dataset"], str(payload["example_id"])) not in selected_keys:
        raise RuntimeError("Saved result falls outside FINAL_EVALUATION boundary.")

completed = {}
for path in sorted(EXAMPLE_DIR.glob("*.json")):
    payload = json.loads(path.read_text(encoding="utf-8"))
    validate_saved_result(payload)
    completed[(payload["dataset"], str(payload["example_id"]))] = payload

# --------------------------------------------------------------------------------------------------
# 11. CLEAN LIVE CONSOLE HELPERS
# --------------------------------------------------------------------------------------------------
def mark(correct):
    return "✓" if correct else "✗"


def fmt_pred(value):
    value = "None" if value is None else str(value)
    return value if len(value) <= 16 else value[:13] + "..."


def hhmmss(seconds):
    seconds = max(0, int(seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def short_question(question, width=115):
    text = " ".join(str(question).split())
    return text if len(text) <= width else text[: width - 3] + "..."


def system_console(label, row):
    return (
        f"  {label:<10} {mark(row['correct'])}  pred={fmt_pred(row['prediction']):<16} "
        f"P={row['planner_steps']:>2} C={row['critic_steps']:>2} S={row['solver_steps']:>2} | "
        f"steps={row['total_latent_steps']:>2} | out_tok={row['generated_tokens']:>3} | "
        f"prompt_tok={row['prompt_tokens_total']:>4} | {row['wall_latency_seconds']:>6.2f}s | "
        f"GPU={row['peak_gpu_memory_gb']:.2f}GB"
    )


def running_accuracy(all_payloads, system):
    rows = [p["systems"][system]["correct"] for p in all_payloads]
    return 100.0 * sum(rows) / len(rows) if rows else 0.0


def create_progress_archive():
    tmp_base = WORKING_ROOT / "adaptive-latent-mas_phase9_progress"
    archive = Path(shutil.make_archive(str(tmp_base), "zip", root_dir=RUN_DIR))
    if archive != PROGRESS_ARCHIVE:
        if PROGRESS_ARCHIVE.exists():
            PROGRESS_ARCHIVE.unlink()
        archive.replace(PROGRESS_ARCHIVE)
    return PROGRESS_ARCHIVE

# --------------------------------------------------------------------------------------------------
# 12. FINAL BENCHMARK LOOP
# --------------------------------------------------------------------------------------------------
print("=" * 112)
print("PHASE 9 — FINAL END-TO-END ADAPTIVE REASONING BENCHMARK")
print("=" * 112)
print("Phase 8 controller       :", PHASE8_MANIFEST["selected_model"])
print("Controller threshold      :", CONTROLLER_THRESHOLD)
print("Backbone                  :", MODEL_ID)
print("Pinned revision           :", MODEL_REVISION)
print("Continuation validation   : PASS")
print("GSM8K FINAL_EVALUATION    :", len(selected_gsm))
print("ARC FINAL_EVALUATION      :", len(selected_arc))
print("GSM8K SEALED_REMAINDER    :", len(sealed_gsm))
print("ARC SEALED_REMAINDER      :", len(sealed_arc))
print("Already completed         :", len(completed))
print("Status command            : python", STATUS_SCRIPT)
print("=" * 112)

benchmark_start = time.perf_counter()
newly_completed = 0
recent_times = []

# Keep execution-order effects from always favoring the same system.
cyclic_orders = [
    ["fixed_10", "fixed_20", "adaptive"],
    ["fixed_20", "adaptive", "fixed_10"],
    ["adaptive", "fixed_10", "fixed_20"],
]

for ordinal, record in enumerate(selected_records, start=1):
    key = (record["dataset"], str(record["id"]))
    if key in completed:
        continue

    problem_start = time.perf_counter()
    order_hash = int(hashlib.sha256(f"ORDER::{record['dataset']}::{record['id']}".encode()).hexdigest()[:8], 16)
    execution_order = cyclic_orders[order_hash % len(cyclic_orders)]
    system_results = {}

    try:
        for system_name in execution_order:
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

            if system_name == "fixed_10":
                row = run_fixed(fixed10_runner, record, 10)
            elif system_name == "fixed_20":
                row = run_fixed(fixed20_runner, record, 20)
            elif system_name == "adaptive":
                row = run_adaptive(record)
            else:
                raise ValueError(system_name)

            row["peak_gpu_memory_gb"] = float(torch.cuda.max_memory_allocated() / (1024 ** 3))
            system_results[system_name] = row

        payload = {
            "phase": 9,
            "phase9_version": PHASE9_VERSION,
            "dataset": record["dataset"],
            "example_id": str(record["id"]),
            "source_split": "test",
            "boundary_label": "FINAL_EVALUATION",
            "question": record["question"],
            "reference_answer": record["reference_answer"],
            "systems": system_results,
            "execution_order": execution_order,
            "controller_model": PHASE8_MANIFEST["selected_model"],
            "controller_threshold": CONTROLLER_THRESHOLD,
            "reference_used_by_controller": False,
            "test_result_used_for_tuning": False,
            "created_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }
        validate_saved_result(payload)
        atomic_json(result_path(record), payload)
        completed[key] = payload

    except Exception:
        progress_payload = {
            "status": "STOPPED_WITH_ERROR",
            "completed": len(completed),
            "total": len(selected_records),
            "current_dataset": record["dataset"],
            "current_example_id": str(record["id"]),
        }
        atomic_json(RUN_DIR / "progress.json", progress_payload)
        create_progress_archive()
        print("\n" + "=" * 112)
        print("PHASE 9 STOPPED SAFELY — COMPLETED RESULTS ARE PRESERVED")
        print("Dataset   :", record["dataset"])
        print("Example ID:", record["id"])
        print("Progress ZIP:", PROGRESS_ARCHIVE)
        print("Attach the progress ZIP and rerun this same Phase-9 cell to resume.")
        print("=" * 112)
        raise

    problem_seconds = time.perf_counter() - problem_start
    recent_times.append(problem_seconds)
    recent_times = recent_times[-20:]
    newly_completed += 1
    done = len(completed)
    remaining = len(selected_records) - done
    mean_recent = float(np.mean(recent_times))
    eta_seconds = mean_recent * remaining
    elapsed_seconds = time.perf_counter() - benchmark_start

    all_payloads = list(completed.values())
    acc = {name: running_accuracy(all_payloads, name) for name in ["fixed_10", "fixed_20", "adaptive"]}

    f10 = system_results["fixed_10"]
    f20 = system_results["fixed_20"]
    ada = system_results["adaptive"]

    print()
    print("─" * 112)
    print(f"[{done:03d}/{len(selected_records)}] {record['dataset'].upper()} | {record['id']} | FINAL_EVALUATION")
    print("Question:", short_question(record["question"]))
    print(system_console("FIXED-10", f10))
    print(system_console("FIXED-20", f20))
    print(system_console("ADAPTIVE", ada))
    print(
        "  Controller  "
        f"P:{ada['planner_decision']['decision']:<8}({ada['planner_decision']['predicted_marginal_utility']:+.4f})  "
        f"C:{ada['critic_decision']['decision']:<8}({ada['critic_decision']['predicted_marginal_utility']:+.4f})  "
        f"S:{ada['solver_decision']['decision']:<8}({ada['solver_decision']['predicted_marginal_utility']:+.4f})"
    )
    print(
        f"  Adaptive    tier={ada['compute_tier']:<6} | saving_vs_F20={ada['latent_step_saving_vs_fixed20_percent']:.2f}% | "
        f"controller={ada['controller_latency_seconds']*1000:.2f}ms"
    )
    print(
        f"  Running accuracy  F10={acc['fixed_10']:.2f}% | F20={acc['fixed_20']:.2f}% | Adaptive={acc['adaptive']:.2f}%"
    )
    print(
        f"  Progress    problem={problem_seconds:.2f}s | elapsed={hhmmss(elapsed_seconds)} | ETA={hhmmss(eta_seconds)}"
    )

    progress_payload = {
        "status": "RUNNING",
        "completed": done,
        "total": len(selected_records),
        "current_dataset": record["dataset"],
        "current_example_id": str(record["id"]),
        "elapsed_hours": round(elapsed_seconds / 3600.0, 3),
        "eta_hours": round(eta_seconds / 3600.0, 3),
        "running_accuracy": {k: round(v, 4) for k, v in acc.items()},
        "latest_progress_archive": str(PROGRESS_ARCHIVE),
    }
    atomic_json(RUN_DIR / "progress.json", progress_payload)

    if newly_completed % CHECKPOINT_EVERY == 0:
        create_progress_archive()
        print("  CHECKPOINT  :", PROGRESS_ARCHIVE)

# --------------------------------------------------------------------------------------------------
# 13. LOAD AND VERIFY ALL 512 FINAL RESULTS
# --------------------------------------------------------------------------------------------------
all_results = []
for record in selected_records:
    path = result_path(record)
    if not path.exists():
        raise RuntimeError(f"Missing Phase-9 result for {record['dataset']} / {record['id']}")
    payload = json.loads(path.read_text(encoding="utf-8"))
    validate_saved_result(payload)
    all_results.append(payload)

if len(all_results) != FINAL_N_PER_DATASET * 2:
    raise RuntimeError("Phase 9 did not complete all 512 FINAL_EVALUATION problems.")

# Assert SEALED_REMAINDER was never written as a result.
for payload in all_results:
    if (payload["dataset"], str(payload["example_id"])) in sealed_keys:
        raise RuntimeError("SEALED_REMAINDER test record was evaluated.")

# --------------------------------------------------------------------------------------------------
# 14. FLAT PREDICTION TABLE
# --------------------------------------------------------------------------------------------------
flat_rows = []
for payload in all_results:
    for system_name, row in payload["systems"].items():
        flat_rows.append({
            "dataset": payload["dataset"],
            "example_id": payload["example_id"],
            "system": system_name,
            "prediction": row["prediction"],
            "reference": row["reference"],
            "correct": bool(row["correct"]),
            "planner_steps": int(row["planner_steps"]),
            "critic_steps": int(row["critic_steps"]),
            "solver_steps": int(row["solver_steps"]),
            "total_latent_steps": int(row["total_latent_steps"]),
            "generated_tokens": int(row["generated_tokens"]),
            "prompt_tokens_total": int(row["prompt_tokens_total"]),
            "wall_latency_seconds": float(row["wall_latency_seconds"]),
            "model_component_latency_seconds": float(row["model_component_latency_seconds"]),
            "peak_gpu_memory_gb": float(row["peak_gpu_memory_gb"]),
            "compute_tier": row["compute_tier"],
            "latent_step_saving_vs_fixed20_percent": float(row["latent_step_saving_vs_fixed20_percent"]),
        })
results_df = pd.DataFrame(flat_rows)
if len(results_df) != 1536:
    raise RuntimeError("Expected 1,536 system-level results.")

# --------------------------------------------------------------------------------------------------
# 15. WILSON ACCURACY INTERVAL
# --------------------------------------------------------------------------------------------------
def wilson_interval(successes, total, z=1.959963984540054):
    if total <= 0:
        return (0.0, 0.0)
    p = successes / total
    denom = 1 + z*z/total
    center = (p + z*z/(2*total)) / denom
    spread = z * math.sqrt((p*(1-p) + z*z/(4*total))/total) / denom
    return (max(0.0, center-spread), min(1.0, center+spread))

summary_rows = []
for dataset in ["gsm8k", "arc_challenge", "combined"]:
    subset = results_df if dataset == "combined" else results_df[results_df["dataset"] == dataset]
    for system in ["fixed_10", "fixed_20", "adaptive"]:
        frame = subset[subset["system"] == system]
        n = len(frame)
        correct_n = int(frame["correct"].sum())
        ci_low, ci_high = wilson_interval(correct_n, n)
        summary_rows.append({
            "dataset": dataset,
            "system": system,
            "n": n,
            "correct_n": correct_n,
            "accuracy": float(correct_n / n),
            "accuracy_ci95_low": float(ci_low),
            "accuracy_ci95_high": float(ci_high),
            "mean_latent_steps": float(frame["total_latent_steps"].mean()),
            "median_latent_steps": float(frame["total_latent_steps"].median()),
            "mean_generated_tokens": float(frame["generated_tokens"].mean()),
            "mean_prompt_tokens": float(frame["prompt_tokens_total"].mean()),
            "mean_wall_latency_seconds": float(frame["wall_latency_seconds"].mean()),
            "median_wall_latency_seconds": float(frame["wall_latency_seconds"].median()),
            "mean_peak_gpu_memory_gb": float(frame["peak_gpu_memory_gb"].mean()),
            "latent_step_reduction_vs_fixed20_percent": float(100.0 * (60.0 - frame["total_latent_steps"].mean()) / 60.0),
        })
summary_df = pd.DataFrame(summary_rows)

# --------------------------------------------------------------------------------------------------
# 16. ADAPTIVE CONTROLLER DECISION TABLE + COMPUTE TIER DISTRIBUTION
# --------------------------------------------------------------------------------------------------
decision_rows = []
tier_rows = []
for payload in all_results:
    adaptive = payload["systems"]["adaptive"]
    for role in ["planner", "critic", "solver"]:
        decision = adaptive[f"{role}_decision"]
        decision_rows.append({
            "dataset": payload["dataset"],
            "example_id": payload["example_id"],
            "agent_role": role,
            "decision": decision["decision"],
            "continued": int(decision["decision"] == "CONTINUE"),
            "predicted_marginal_utility": float(decision["predicted_marginal_utility"]),
            "selected_steps": int(adaptive[f"{role}_steps"]),
        })
    tier_rows.append({
        "dataset": payload["dataset"],
        "example_id": payload["example_id"],
        "compute_tier": adaptive["compute_tier"],
        "total_latent_steps": adaptive["total_latent_steps"],
    })

decision_df = pd.DataFrame(decision_rows)
decision_summary_df = (
    decision_df.groupby(["dataset", "agent_role"], as_index=False)
    .agg(
        n=("continued", "size"),
        continue_rate=("continued", "mean"),
        mean_selected_steps=("selected_steps", "mean"),
        mean_predicted_utility=("predicted_marginal_utility", "mean"),
    )
)
decision_summary_df["stop_rate"] = 1.0 - decision_summary_df["continue_rate"]

tier_df = pd.DataFrame(tier_rows)
tier_summary_df = (
    tier_df.groupby(["dataset", "compute_tier"], as_index=False)
    .agg(n=("example_id", "size"), mean_steps=("total_latent_steps", "mean"))
)

# --------------------------------------------------------------------------------------------------
# 17. EXACT McNEMAR + PROBLEM-LEVEL BOOTSTRAP CONFIDENCE INTERVALS
# --------------------------------------------------------------------------------------------------
def paired_frame(dataset, baseline):
    subset = results_df if dataset == "combined" else results_df[results_df["dataset"] == dataset]
    adaptive = subset[subset["system"] == "adaptive"][
        ["dataset", "example_id", "correct", "total_latent_steps", "wall_latency_seconds", "generated_tokens"]
    ].rename(columns={
        "correct": "adaptive_correct",
        "total_latent_steps": "adaptive_steps",
        "wall_latency_seconds": "adaptive_latency",
        "generated_tokens": "adaptive_tokens",
    })
    base = subset[subset["system"] == baseline][
        ["dataset", "example_id", "correct", "total_latent_steps", "wall_latency_seconds", "generated_tokens"]
    ].rename(columns={
        "correct": "baseline_correct",
        "total_latent_steps": "baseline_steps",
        "wall_latency_seconds": "baseline_latency",
        "generated_tokens": "baseline_tokens",
    })
    return adaptive.merge(base, on=["dataset", "example_id"], validate="one_to_one")


def bootstrap_paired(frame, dataset, seed_offset):
    rng = np.random.default_rng(SEED + seed_offset)
    metrics = {"accuracy_diff": [], "step_saving": [], "latency_diff": [], "token_diff": []}

    if dataset == "combined":
        groups = {
            d: frame[frame["dataset"] == d].reset_index(drop=True)
            for d in ["gsm8k", "arc_challenge"]
        }
        samples = []
        for _ in range(BOOTSTRAP_REPLICATES):
            pieces = []
            for d, g in groups.items():
                idx = rng.integers(0, len(g), size=len(g))
                pieces.append(g.iloc[idx])
            samples.append(pd.concat(pieces, ignore_index=True))
    else:
        g = frame.reset_index(drop=True)
        samples = [g.iloc[rng.integers(0, len(g), size=len(g))] for _ in range(BOOTSTRAP_REPLICATES)]

    for sample in samples:
        metrics["accuracy_diff"].append(float(sample["adaptive_correct"].mean() - sample["baseline_correct"].mean()))
        metrics["step_saving"].append(float(sample["baseline_steps"].mean() - sample["adaptive_steps"].mean()))
        metrics["latency_diff"].append(float(sample["baseline_latency"].mean() - sample["adaptive_latency"].mean()))
        metrics["token_diff"].append(float(sample["baseline_tokens"].mean() - sample["adaptive_tokens"].mean()))

    return {
        name: {
            "ci95_low": float(np.percentile(values, 2.5)),
            "ci95_high": float(np.percentile(values, 97.5)),
        }
        for name, values in metrics.items()
    }

paired_rows = []
comparison_counter = 0
for dataset in ["gsm8k", "arc_challenge", "combined"]:
    for baseline in ["fixed_10", "fixed_20"]:
        comparison_counter += 1
        frame = paired_frame(dataset, baseline)
        a = frame["adaptive_correct"].astype(bool)
        b = frame["baseline_correct"].astype(bool)
        adaptive_only = int((a & ~b).sum())
        baseline_only = int((~a & b).sum())
        discordant = adaptive_only + baseline_only
        p_value = 1.0 if discordant == 0 else float(binomtest(adaptive_only, discordant, p=0.5).pvalue)
        boot = bootstrap_paired(frame, dataset, 900 + comparison_counter)

        paired_rows.append({
            "dataset": dataset,
            "comparison": f"adaptive_vs_{baseline}",
            "n": len(frame),
            "adaptive_accuracy": float(a.mean()),
            "baseline_accuracy": float(b.mean()),
            "accuracy_difference": float(a.mean() - b.mean()),
            "accuracy_difference_ci95_low": boot["accuracy_diff"]["ci95_low"],
            "accuracy_difference_ci95_high": boot["accuracy_diff"]["ci95_high"],
            "adaptive_only_correct": adaptive_only,
            "baseline_only_correct": baseline_only,
            "both_correct": int((a & b).sum()),
            "both_wrong": int((~a & ~b).sum()),
            "mcnemar_exact_p": p_value,
            "mcnemar_significant_0_05": bool(p_value < 0.05),
            "mean_latent_steps_saved": float(frame["baseline_steps"].mean() - frame["adaptive_steps"].mean()),
            "latent_steps_saved_ci95_low": boot["step_saving"]["ci95_low"],
            "latent_steps_saved_ci95_high": boot["step_saving"]["ci95_high"],
            "relative_latent_step_change_percent": float(
                100.0 * (frame["baseline_steps"].mean() - frame["adaptive_steps"].mean()) / frame["baseline_steps"].mean()
            ),
            "mean_latency_difference_seconds": float(frame["baseline_latency"].mean() - frame["adaptive_latency"].mean()),
            "latency_difference_ci95_low": boot["latency_diff"]["ci95_low"],
            "latency_difference_ci95_high": boot["latency_diff"]["ci95_high"],
            "mean_generated_token_difference": float(frame["baseline_tokens"].mean() - frame["adaptive_tokens"].mean()),
            "generated_token_difference_ci95_low": boot["token_diff"]["ci95_low"],
            "generated_token_difference_ci95_high": boot["token_diff"]["ci95_high"],
        })
paired_df = pd.DataFrame(paired_rows)

# --------------------------------------------------------------------------------------------------
# 18. SAVE PREDICTIONS, TABLES AND SUMMARY JSON
# --------------------------------------------------------------------------------------------------
RESULTS_CSV = TABLE_DIR / "phase9_final_system_results.csv"
SUMMARY_CSV = TABLE_DIR / "phase9_final_performance_summary.csv"
DECISIONS_CSV = TABLE_DIR / "phase9_adaptive_decisions.csv"
DECISION_SUMMARY_CSV = TABLE_DIR / "phase9_adaptive_decision_summary.csv"
TIER_SUMMARY_CSV = TABLE_DIR / "phase9_compute_tier_summary.csv"
PAIRED_CSV = TABLE_DIR / "phase9_paired_statistical_comparison.csv"
PREDICTIONS_JSONL = RESULT_FINAL_DIR / "phase9_final_test_predictions.jsonl"
SUMMARY_JSON = RESULT_FINAL_DIR / "phase9_final_summary.json"

results_df.to_csv(RESULTS_CSV, index=False)
summary_df.to_csv(SUMMARY_CSV, index=False)
decision_df.to_csv(DECISIONS_CSV, index=False)
decision_summary_df.to_csv(DECISION_SUMMARY_CSV, index=False)
tier_summary_df.to_csv(TIER_SUMMARY_CSV, index=False)
paired_df.to_csv(PAIRED_CSV, index=False)

with PREDICTIONS_JSONL.open("w", encoding="utf-8") as handle:
    for payload in all_results:
        handle.write(json.dumps(payload, ensure_ascii=False) + "\n")

combined = summary_df[summary_df["dataset"] == "combined"].set_index("system")
main_pair = paired_df[(paired_df["dataset"] == "combined") & (paired_df["comparison"] == "adaptive_vs_fixed_20")].iloc[0]
final_summary_payload = {
    "phase": 9,
    "status": "PHASE_9_COMPLETE",
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "controller": PHASE8_MANIFEST["selected_model"],
    "threshold": CONTROLLER_THRESHOLD,
    "test_problems": {"gsm8k": 256, "arc_challenge": 256, "total": 512},
    "sealed_remainder": {"gsm8k": len(sealed_gsm), "arc_challenge": len(sealed_arc)},
    "continuation_validation": CONTINUATION_VALIDATION,
    "combined": {
        system: {
            key: (int(value) if key in {"n", "correct_n"} else float(value))
            for key, value in combined.loc[system].to_dict().items()
        }
        for system in ["fixed_10", "fixed_20", "adaptive"]
    },
    "adaptive_vs_fixed20": main_pair.to_dict(),
    "controller_retrained": False,
    "threshold_tuned_on_test": False,
    "reference_used_by_controller": False,
    "sealed_remainder_evaluated": False,
}
atomic_json(SUMMARY_JSON, final_summary_payload)

# --------------------------------------------------------------------------------------------------
# 19. PRESENTATION-READY FIGURES
# --------------------------------------------------------------------------------------------------
import matplotlib.pyplot as plt

# Accuracy comparison
plot_df = summary_df[summary_df["dataset"].isin(["gsm8k", "arc_challenge", "combined"])].copy()
pivot = plot_df.pivot(index="dataset", columns="system", values="accuracy") * 100.0
ax = pivot[["fixed_10", "fixed_20", "adaptive"]].plot(kind="bar", figsize=(9, 5))
ax.set_ylabel("Accuracy (%)")
ax.set_xlabel("Dataset")
ax.set_title("Phase 9: Final Accuracy Comparison")
ax.legend(title="System")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "phase9_accuracy_comparison.png", dpi=200)
plt.close()

# Mean latent steps
pivot_steps = plot_df.pivot(index="dataset", columns="system", values="mean_latent_steps")
ax = pivot_steps[["fixed_10", "fixed_20", "adaptive"]].plot(kind="bar", figsize=(9, 5))
ax.set_ylabel("Mean latent steps per problem")
ax.set_xlabel("Dataset")
ax.set_title("Phase 9: Latent Computation Comparison")
ax.legend(title="System")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "phase9_latent_steps_comparison.png", dpi=200)
plt.close()

# Adaptive continuation rates
pivot_decisions = decision_summary_df.pivot(index="agent_role", columns="dataset", values="continue_rate") * 100.0
ax = pivot_decisions.plot(kind="bar", figsize=(8, 5))
ax.set_ylabel("CONTINUE decisions (%)")
ax.set_xlabel("Agent role")
ax.set_title("Phase 9: Adaptive Controller Continuation Rate")
ax.legend(title="Dataset")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "phase9_adaptive_continue_rate.png", dpi=200)
plt.close()

# Accuracy-compute tradeoff
fig, ax = plt.subplots(figsize=(8, 5))
for system in ["fixed_10", "fixed_20", "adaptive"]:
    row = combined.loc[system]
    ax.scatter(float(row["mean_latent_steps"]), float(row["accuracy"]) * 100.0, s=90, label=system)
    ax.annotate(system, (float(row["mean_latent_steps"]), float(row["accuracy"]) * 100.0), xytext=(5, 5), textcoords="offset points")
ax.set_xlabel("Mean latent steps per problem")
ax.set_ylabel("Accuracy (%)")
ax.set_title("Phase 9: Accuracy–Compute Trade-off")
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "phase9_accuracy_compute_tradeoff.png", dpi=200)
plt.close()

# Latency comparison
pivot_latency = plot_df.pivot(index="dataset", columns="system", values="mean_wall_latency_seconds")
ax = pivot_latency[["fixed_10", "fixed_20", "adaptive"]].plot(kind="bar", figsize=(9, 5))
ax.set_ylabel("Mean wall latency (seconds)")
ax.set_xlabel("Dataset")
ax.set_title("Phase 9: End-to-End Latency Comparison")
ax.legend(title="System")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "phase9_latency_comparison.png", dpi=200)
plt.close()

# --------------------------------------------------------------------------------------------------
# 20. FINAL MANIFEST + SHA-256
# --------------------------------------------------------------------------------------------------
PHASE9_MANIFEST_PATH = SUMMARY_DIR / "phase9_final_manifest.json"
phase9_manifest = {
    "phase": 9,
    "title": "Final End-to-End Adaptive Reasoning Benchmark",
    "status": "PHASE_9_COMPLETE",
    "phase9_version": PHASE9_VERSION,
    "seed": SEED,
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "controller": PHASE8_MANIFEST["selected_model"],
    "controller_threshold": CONTROLLER_THRESHOLD,
    "controller_retrained": False,
    "threshold_tuned_on_test": False,
    "feature_selection_on_test": False,
    "reference_used_by_controller": False,
    "official_test_used": True,
    "official_validation_used": False,
    "true_latent_continuation": True,
    "continuation_validation": CONTINUATION_VALIDATION,
    "final_evaluation_counts": {"gsm8k": 256, "arc_challenge": 256, "total": 512},
    "sealed_remainder_counts": {"gsm8k": len(sealed_gsm), "arc_challenge": len(sealed_arc)},
    "systems": {
        "fixed_10": "Planner 10 / Critic 10 / Solver 10",
        "fixed_20": "Planner 20 / Critic 20 / Solver 20",
        "adaptive": "Independent learned STOP at 10 or CONTINUE to 20 per agent",
    },
    "statistics": {
        "accuracy_interval": "Wilson 95%",
        "paired_accuracy_test": "Exact McNemar/binomial",
        "paired_bootstrap_replicates": BOOTSTRAP_REPLICATES,
    },
    "files": {},
}

hash_targets = [
    RESULTS_CSV, SUMMARY_CSV, DECISIONS_CSV, DECISION_SUMMARY_CSV, TIER_SUMMARY_CSV,
    PAIRED_CSV, PREDICTIONS_JSONL, SUMMARY_JSON, BOUNDARY_PATH,
    SUMMARY_DIR / "phase9_continuation_validation.json",
    FIGURE_DIR / "phase9_accuracy_comparison.png",
    FIGURE_DIR / "phase9_latent_steps_comparison.png",
    FIGURE_DIR / "phase9_adaptive_continue_rate.png",
    FIGURE_DIR / "phase9_accuracy_compute_tradeoff.png",
    FIGURE_DIR / "phase9_latency_comparison.png",
    CONTROLLER_PATH,
]
for path in hash_targets:
    phase9_manifest["files"][str(path.relative_to(PROJECT_ROOT))] = {
        "sha256": sha256_file(path),
        "bytes": int(path.stat().st_size),
    }
atomic_json(PHASE9_MANIFEST_PATH, phase9_manifest)

# --------------------------------------------------------------------------------------------------
# 21. PHASE-9 REGRESSION TESTS
# --------------------------------------------------------------------------------------------------
PHASE9_TEST = TEST_DIR / "test_phase9_final_evaluation.py"
PHASE9_TEST.write_text(r'''from pathlib import Path
import json
import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
MANIFEST = ROOT / "experiments/summaries/phase9_final_manifest.json"
RESULTS = ROOT / "results/tables/phase9_final_system_results.csv"
BOUNDARIES = ROOT / "experiments/runs/phase9_final_evaluation/dataset_boundaries.json"


def manifest():
    assert MANIFEST.exists()
    return json.loads(MANIFEST.read_text(encoding="utf-8"))


def test_phase9_complete():
    assert manifest()["status"] == "PHASE_9_COMPLETE"


def test_frozen_controller_protocol():
    row = manifest()
    assert row["controller_retrained"] is False
    assert row["threshold_tuned_on_test"] is False
    assert row["feature_selection_on_test"] is False
    assert row["reference_used_by_controller"] is False


def test_true_continuation_validated():
    row = manifest()
    assert row["true_latent_continuation"] is True
    assert row["continuation_validation"]["passed"] is True


def test_final_counts():
    row = manifest()
    assert row["final_evaluation_counts"] == {"gsm8k": 256, "arc_challenge": 256, "total": 512}


def test_three_systems_and_rows():
    frame = pd.read_csv(RESULTS)
    assert len(frame) == 1536
    assert set(frame["system"].unique()) == {"fixed_10", "fixed_20", "adaptive"}


def test_fixed_step_counts():
    frame = pd.read_csv(RESULTS)
    assert (frame.loc[frame.system == "fixed_10", "total_latent_steps"] == 30).all()
    assert (frame.loc[frame.system == "fixed_20", "total_latent_steps"] == 60).all()
    assert set(frame.loc[frame.system == "adaptive", "total_latent_steps"].unique()).issubset({30,40,50,60})


def test_boundaries_exist_and_remainder_sealed():
    row = json.loads(BOUNDARIES.read_text(encoding="utf-8"))
    assert row["final_evaluation_counts"] == {"gsm8k": 256, "arc_challenge": 256}
    assert row["sealed_remainder_counts"]["gsm8k"] > 0
    assert row["sealed_remainder_counts"]["arc_challenge"] > 0
''', encoding="utf-8")

# --------------------------------------------------------------------------------------------------
# 22. UPDATE PROJECT STATUS + RUN COMPLETE TEST SUITE
# --------------------------------------------------------------------------------------------------
STATUS_PATH = PROJECT_ROOT / "docs/STATUS.md"
status_marker = "## Phase 9 — Final End-to-End Evaluation: COMPLETE"
status_text = STATUS_PATH.read_text(encoding="utf-8") if STATUS_PATH.exists() else ""
if status_marker not in status_text:
    with STATUS_PATH.open("a", encoding="utf-8") as handle:
        handle.write("\n\n" + status_marker + "\n")
        handle.write("- Frozen learned controller evaluated on 512 unseen official-test problems.\n")
        handle.write("- Fixed-10, Fixed-20 and Adaptive systems compared.\n")
        handle.write("- True 10→20 continuation validated and used.\n")
        handle.write("- Wilson intervals, exact McNemar tests and paired bootstrap intervals reported.\n")
        handle.write("- SEALED_REMAINDER official-test records were not evaluated.\n")

PROJECT_YAML = PROJECT_ROOT / "project.yaml"
project_cfg = yaml.safe_load(PROJECT_YAML.read_text(encoding="utf-8"))
project_cfg["project"]["version"] = "0.9.0"
project_cfg["project"]["stage"] = "phase_9_final_evaluation_complete"
project_cfg["research"]["contributions"]["adaptive_latent_steps"]["status"] = "evaluated"
project_cfg["adaptive_computation"]["phase8_learned_controller"] = {
    "status": "trained_and_frozen",
    "type": "gradient_boosting_huber",
    "decision": "stop_at_10_or_continue_to_20",
    "phase9_final_evaluation": "complete",
    "test_set_used_for_tuning": False,
}
PROJECT_YAML.write_text(yaml.safe_dump(project_cfg, sort_keys=False), encoding="utf-8")

test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)
print("\n" + "=" * 112)
print("PROJECT REGRESSION TESTS")
print("=" * 112)
print(test_result.stdout)
if test_result.stderr:
    print(test_result.stderr)
if test_result.returncode != 0:
    phase9_manifest["status"] = "PHASE_9_TEST_FAILED"
    atomic_json(PHASE9_MANIFEST_PATH, phase9_manifest)
    raise RuntimeError("Phase 9 regression tests failed.")

# --------------------------------------------------------------------------------------------------
# 23. FINAL FULL + LIGHTWEIGHT + PROGRESS ARCHIVES
# --------------------------------------------------------------------------------------------------
create_progress_archive()
FULL_BACKUP = Path(shutil.make_archive(
    str(WORKING_ROOT / "adaptive-latent-mas_phase9_complete"),
    "zip",
    root_dir=PROJECT_ROOT,
))

RELEASE_DIR = WORKING_ROOT / "phase9_release"
if RELEASE_DIR.exists():
    shutil.rmtree(RELEASE_DIR)
RELEASE_DIR.mkdir(parents=True)
for rel in [
    "src", "configs", "scripts", "tests", "docs", "project.yaml", "pyproject.toml", "requirements.txt",
    "experiments/checkpoints", "experiments/summaries", "results/tables", "results/figures", "results/final",
]:
    source = PROJECT_ROOT / rel
    destination = RELEASE_DIR / rel
    if not source.exists():
        continue
    destination.parent.mkdir(parents=True, exist_ok=True)
    if source.is_dir():
        shutil.copytree(source, destination)
    else:
        shutil.copy2(source, destination)
LIGHT_BACKUP = Path(shutil.make_archive(
    str(WORKING_ROOT / "adaptive-latent-mas_phase9_release"),
    "zip",
    root_dir=RELEASE_DIR,
))
shutil.rmtree(RELEASE_DIR)

# --------------------------------------------------------------------------------------------------
# 24. FINAL REPORT
# --------------------------------------------------------------------------------------------------
atomic_json(RUN_DIR / "progress.json", {
    "status": "PHASE_9_COMPLETE",
    "completed": 512,
    "total": 512,
    "elapsed_hours": round((time.perf_counter() - benchmark_start) / 3600.0, 3),
    "running_accuracy": {
        system: round(float(combined.loc[system, "accuracy"] * 100), 4)
        for system in ["fixed_10", "fixed_20", "adaptive"]
    },
})

print("\n" + "=" * 112)
print("PHASE 9 COMPLETE — FINAL END-TO-END ADAPTIVE REASONING BENCHMARK")
print("=" * 112)
print("\nFINAL PERFORMANCE SUMMARY")
display(summary_df[[
    "dataset", "system", "n", "correct_n", "accuracy", "accuracy_ci95_low", "accuracy_ci95_high",
    "mean_latent_steps", "latent_step_reduction_vs_fixed20_percent", "mean_generated_tokens",
    "mean_wall_latency_seconds", "mean_peak_gpu_memory_gb",
]])
print("\nADAPTIVE CONTROLLER DECISIONS")
display(decision_summary_df)
print("\nCOMPUTE TIER DISTRIBUTION")
display(tier_summary_df)
print("\nPAIRED STATISTICAL COMPARISON")
display(paired_df[[
    "dataset", "comparison", "n", "adaptive_accuracy", "baseline_accuracy", "accuracy_difference",
    "accuracy_difference_ci95_low", "accuracy_difference_ci95_high", "mcnemar_exact_p",
    "mean_latent_steps_saved", "relative_latent_step_change_percent",
    "mean_latency_difference_seconds",
]])

print("\nFINAL COMBINED RESULT")
for system in ["fixed_10", "fixed_20", "adaptive"]:
    row = combined.loc[system]
    print(
        f"{system:<10}: accuracy={row['accuracy']*100:.2f}% "
        f"[{row['accuracy_ci95_low']*100:.2f}, {row['accuracy_ci95_high']*100:.2f}] | "
        f"steps={row['mean_latent_steps']:.2f} | tokens={row['mean_generated_tokens']:.2f} | "
        f"latency={row['mean_wall_latency_seconds']:.2f}s"
    )

print("\nAdaptive vs Fixed-20")
print(f"Accuracy difference         : {main_pair['accuracy_difference']*100:+.2f} percentage points")
print(f"95% bootstrap interval      : [{main_pair['accuracy_difference_ci95_low']*100:+.2f}, {main_pair['accuracy_difference_ci95_high']*100:+.2f}]")
print(f"Exact McNemar p-value       : {main_pair['mcnemar_exact_p']:.6f}")
print(f"Mean latent steps saved     : {main_pair['mean_latent_steps_saved']:.2f}")
print(f"Relative latent-step change : {main_pair['relative_latent_step_change_percent']:.2f}%")
print(f"Mean latency difference     : {main_pair['mean_latency_difference_seconds']:+.3f}s (positive = Adaptive faster)")

print("\nRESEARCH INTEGRITY")
print("Controller retrained        : NO")
print("Threshold tuned on test     : NO")
print("Reference used by controller: NO")
print("SEALED_REMAINDER evaluated  : NO")
print("True 10→20 continuation     : VERIFIED")
print("Regression tests            : PASS")

print("\nOUTPUTS")
print("Predictions                  :", PREDICTIONS_JSONL)
print("Performance summary          :", SUMMARY_CSV)
print("Paired statistics            :", PAIRED_CSV)
print("SHA manifest                 :", PHASE9_MANIFEST_PATH)
print("Progress/recovery ZIP        :", PROGRESS_ARCHIVE)
print("Full Phase-9 backup          :", FULL_BACKUP)
print("Lightweight Phase-9 release  :", LIGHT_BACKUP)
print("Status command               : python", STATUS_SCRIPT)
print("=" * 112)
print("PHASE 9 STATUS: COMPLETE")
print("=" * 112)


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (3 outputs left out)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

PHASE 9 — FINAL END-TO-END ADAPTIVE REASONING BENCHMARK
Phase 8 controller       : gradient_boosting_huber
Controller threshold      : 0.0
Backbone                  : Qwen/Qwen3-4B
Pinned revision           : 1cfa9a7208912126459214e8b04321603b3df60c
Continuation validation   : PASS
GSM8K FINAL_EVALUATION    : 256
ARC FINAL_EVALUATION      : 256
GSM8K SEALED_REMAINDER    : 1063
ARC SEALED_REMAINDER      : 916
Already completed         : 0
Status command            : python /kaggle/working/adaptive-latent-mas/scripts/phase9_status.py

────────────────────────────────────────────────────────────────────────────────────────────────────────────────
[001/512] ARC_CHALLENGE | arc_challenge_MCAS_2014_8_16 | FINAL_EVALUATION
Question: Just before sunset, a student recorded the temperature of the air and the soil at the same location. The student...
  FIXED-10   ✓  pred=D                P=10 C=10 S=10 | steps=30 | out_tok=  6 | prompt_tok= 571 |   4.02s | GPU=7.73GB
  FIXED-20   ✓  pred=D       

ValueError: could not convert string to float: 'combined'

In [9]:
# =============================================================================
# PHASE 9 — FINALIZATION RECOVERY
# NO MODEL INFERENCE / NO 512-PROBLEM RERUN
# =============================================================================

print("=" * 100)
print("PHASE 9 FINALIZATION RECOVERY")
print("=" * 100)

# -------------------------------------------------------------------------
# 1. FIRST PROTECT ALL 512 SAVED RESULTS
# -------------------------------------------------------------------------
safety_archive = create_progress_archive()

print("512-result recovery archive :", safety_archive)
print("Benchmark inference rerun   : NO")

# -------------------------------------------------------------------------
# 2. REBUILD FINAL SUMMARY JSON WITH CORRECT TYPE HANDLING
# -------------------------------------------------------------------------
combined = (
    summary_df[
        summary_df["dataset"] == "combined"
    ]
    .set_index("system")
)

main_pair = paired_df[
    (paired_df["dataset"] == "combined")
    &
    (paired_df["comparison"] == "adaptive_vs_fixed_20")
].iloc[0]

final_summary_payload = {
    "phase": 9,
    "status": "PHASE_9_COMPLETE",
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "controller": PHASE8_MANIFEST["selected_model"],
    "threshold": CONTROLLER_THRESHOLD,

    "test_problems": {
        "gsm8k": 256,
        "arc_challenge": 256,
        "total": 512,
    },

    "sealed_remainder": {
        "gsm8k": len(sealed_gsm),
        "arc_challenge": len(sealed_arc),
    },

    "continuation_validation": CONTINUATION_VALIDATION,

    # FIX:
    # preserve string columns such as dataset="combined"
    # instead of trying float("combined")
    "combined": {
        system: {
            key: (
                int(value)
                if key in {"n", "correct_n"}
                else value
            )
            for key, value
            in combined.loc[system].to_dict().items()
        }
        for system in [
            "fixed_10",
            "fixed_20",
            "adaptive",
        ]
    },

    "adaptive_vs_fixed20": main_pair.to_dict(),

    "controller_retrained": False,
    "threshold_tuned_on_test": False,
    "reference_used_by_controller": False,
    "sealed_remainder_evaluated": False,
}

atomic_json(
    SUMMARY_JSON,
    final_summary_payload,
)

print("Final summary JSON          : FIXED")

# -------------------------------------------------------------------------
# 3. CONTINUE ONLY FROM SECTION 19 OF THE ORIGINAL PHASE-9 CELL
# -------------------------------------------------------------------------
phase9_source = next(
    source
    for source in reversed(In[:-1])
    if "# 19. PRESENTATION-READY FIGURES" in source
    and "# 24. FINAL REPORT" in source
)

start_marker = "# 19. PRESENTATION-READY FIGURES"

remaining_phase9_code = phase9_source[
    phase9_source.index(start_marker):
]

print("Continuing finalization     : sections 19–24")
print("512-problem benchmark       : WILL NOT RUN AGAIN")
print("=" * 100)

exec(
    remaining_phase9_code,
    globals(),
)

PHASE 9 FINALIZATION RECOVERY
512-result recovery archive : /kaggle/working/adaptive-latent-mas_phase9_progress.zip
Benchmark inference rerun   : NO
Final summary JSON          : FIXED
Continuing finalization     : sections 19–24
512-problem benchmark       : WILL NOT RUN AGAIN

PROJECT REGRESSION TESTS
........................................................................ [100%]
72 passed in 1.33s


PHASE 9 COMPLETE — FINAL END-TO-END ADAPTIVE REASONING BENCHMARK

FINAL PERFORMANCE SUMMARY


,dataset,system,n,correct_n,accuracy,accuracy_ci95_low,accuracy_ci95_high,mean_latent_steps,latent_step_reduction_vs_fixed20_percent,mean_generated_tokens,mean_wall_latency_seconds,mean_peak_gpu_memory_gb
0,gsm8k,fixed_10,256,154,0.601562,0.540515,0.659607,30.000000,50.000000,128.589844,10.514111,7.680603
1,gsm8k,fixed_20,256,109,0.425781,0.366748,0.487009,60.000000,0.000000,130.238281,12.379548,7.685175
2,gsm8k,adaptive,256,139,0.542969,0.481761,0.602906,48.476562,19.205729,127.539062,11.578514,7.683888
3,arc_challenge,fixed_10,256,163,0.636719,0.576185,0.693210,30.000000,50.000000,92.644531,8.571863,7.681075
4,arc_challenge,fixed_20,256,179,0.699219,0.640435,0.752112,60.000000,0.000000,62.968750,8.695916,7.684892
5,arc_challenge,adaptive,256,181,0.707031,0.648548,0.759393,49.609375,17.317708,79.898438,9.034139,7.683798
6,combined,fixed_10,512,317,0.619141,0.576339,0.660168,30.000000,50.000000,110.617188,9.542987,7.680839
7,combined,fixed_20,512,288,0.562500,0.519223,0.604847,60.000000,0.000000,96.603516,10.537732,7.685033
8,combined,adaptive,512,320,0.625000,0.582281,0.665857,49.042969,18.261719,103.718750,10.306326,7.683843



ADAPTIVE CONTROLLER DECISIONS


,dataset,agent_role,n,continue_rate,mean_selected_steps,mean_predicted_utility,stop_rate
0,arc_challenge,critic,256,0.601562,16.015625,0.006030,0.398438
1,arc_challenge,planner,256,0.832031,18.320312,0.020072,0.167969
2,arc_challenge,solver,256,0.527344,15.273438,0.007256,0.472656
3,gsm8k,critic,256,0.613281,16.132812,0.004799,0.386719
4,gsm8k,planner,256,0.769531,17.695312,0.034263,0.230469
5,gsm8k,solver,256,0.464844,14.648438,0.008697,0.535156



COMPUTE TIER DISTRIBUTION


,dataset,compute_tier,n,mean_steps
0,arc_challenge,HIGH,71,60.000000
1,arc_challenge,LOW,6,30.000000
2,arc_challenge,MEDIUM,179,46.145251
3,gsm8k,HIGH,60,60.000000
4,gsm8k,LOW,15,30.000000
5,gsm8k,MEDIUM,181,46.187845



PAIRED STATISTICAL COMPARISON


,dataset,comparison,n,adaptive_accuracy,baseline_accuracy,accuracy_difference,accuracy_difference_ci95_low,accuracy_difference_ci95_high,mcnemar_exact_p,mean_latent_steps_saved,relative_latent_step_change_percent,mean_latency_difference_seconds
0,gsm8k,adaptive_vs_fixed_10,256,0.542969,0.601562,-0.058594,-0.125000,0.003906,0.095924,-18.476562,-61.588542,-1.064403
1,gsm8k,adaptive_vs_fixed_20,256,0.542969,0.425781,0.117188,0.058594,0.179688,0.000287,11.523438,19.205729,0.801034
2,arc_challenge,adaptive_vs_fixed_10,256,0.707031,0.636719,0.070312,0.000000,0.144531,0.059808,-19.609375,-65.364583,-0.462276
3,arc_challenge,adaptive_vs_fixed_20,256,0.707031,0.699219,0.007812,-0.046875,0.062598,0.891923,10.390625,17.317708,-0.338223
4,combined,adaptive_vs_fixed_10,512,0.625000,0.619141,0.005859,-0.041016,0.052734,0.871618,-19.042969,-63.476562,-0.763340
5,combined,adaptive_vs_fixed_20,512,0.625000,0.562500,0.062500,0.021484,0.105469,0.004455,10.957031,18.261719,0.231406



FINAL COMBINED RESULT
fixed_10  : accuracy=61.91% [57.63, 66.02] | steps=30.00 | tokens=110.62 | latency=9.54s
fixed_20  : accuracy=56.25% [51.92, 60.48] | steps=60.00 | tokens=96.60 | latency=10.54s
adaptive  : accuracy=62.50% [58.23, 66.59] | steps=49.04 | tokens=103.72 | latency=10.31s

Adaptive vs Fixed-20
Accuracy difference         : +6.25 percentage points
95% bootstrap interval      : [+2.15, +10.55]
Exact McNemar p-value       : 0.004455
Mean latent steps saved     : 10.96
Relative latent-step change : 18.26%
Mean latency difference     : +0.231s (positive = Adaptive faster)

RESEARCH INTEGRITY
Controller retrained        : NO
Threshold tuned on test     : NO
Reference used by controller: NO
SEALED_REMAINDER evaluated  : NO
True 10→20 continuation     : VERIFIED
Regression tests            : PASS

OUTPUTS
Predictions                  : /kaggle/working/adaptive-latent-mas/results/final/phase9_final_test_predictions.jsonl
Performance summary          : /kaggle/working/adaptive-